In [1]:
import pandas as pd
import os
import joblib
import json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,recall_score, f1_score, confusion_matrix, classification_report)

In [2]:
X_train = pd.read_csv("data/processed/X_train.csv")
y_train = pd.read_csv("data/processed/y_train.csv").squeeze("columns")
X_test = pd.read_csv("data/processed/X_test.csv")
y_test = pd.read_csv("data/processed/y_test.csv").squeeze("columns")

print(X_train.shape, X_test.shape)

(31524, 63) (7880, 63)


In [3]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42))])

In [4]:
pipeline.fit(X_train, y_train)
print("Trained")

Trained


In [5]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

In [6]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.5323604060913706
ROC-AUC: 0.5274666118710251
Precision: 0.3350239504085658
Recall: 0.47295147175815433
F1: 0.3922150750453571


In [7]:
print(classification_report(y_test, y_pred, target_names=["no_deposit", "deposit"]))

              precision    recall  f1-score   support

  no_deposit       0.69      0.56      0.62      5366
     deposit       0.34      0.47      0.39      2514

    accuracy                           0.53      7880
   macro avg       0.51      0.52      0.51      7880
weighted avg       0.58      0.53      0.55      7880



In [8]:
os.makedirs("models", exist_ok=True)
joblib.dump(pipeline, "models/model1_logreg.joblib")
print("Model saved")

Model saved


In [9]:
predictions_df = X_test.copy()
predictions_df["y_true"] = y_test.values
predictions_df["y_pred"] = y_pred
predictions_df["y_proba"] = y_proba
predictions_df[["y_true", "y_pred", "y_proba"]].to_csv(
    "data/processed/model1_predictions.csv", index=False)

metrics = {
    "accuracy": accuracy,
    "roc_auc": roc_auc,
    "precision": precision,
    "recall": recall,
    "f1": f1}
with open("data/processed/model1_metrics.json", "w") as f:
  print("Saved predictions and metrics")

Saved predictions and metrics


In [ ]:
y_t, y_p, y_pr = np.asarray(y_test), np.asarray(y_pred), np.asarray(y_proba)
print(confusion_matrix(y_t, y_p))

rng = np.random.default_rng(42)
auc, rec, f1s = [], [], []
for _ in range(1000):
    i = rng.integers(0, len(y_t), len(y_t))
    auc.append(roc_auc_score(y_t[i], y_pr[i]))
    rec.append(recall_score(y_t[i], y_p[i]))
    f1s.append(f1_score(y_t[i], y_p[i]))

for name, v in [("ROC-AUC", auc), ("Recall", rec), ("F1", f1s)]:
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{name}: 95% CI [{lo:.3f}, {hi:.3f}]")

[[3006 2360]
 [1325 1189]]
